# 2026 World Cup ETL primary process

This first exercise will perform a first extraction to API-Football, ideally we are going to consider the 48 teams that will participate in the world cup and fetch their results on the last 2 years (the time window will be expanded if possible and necessary).

First of all we will try to connect to the API, and get the codes for the national teams

In [9]:
#%pip install -r requirements.txt

In [ ]:
import os
import json
import time
import random
from datetime import datetime
from dotenv import load_dotenv
from libs.api_client import APIFootballClient

# ============================================================
# 0. CONFIGURATION
# ============================================================

load_dotenv()

OUTPUT_DIR = "input"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# Límite diario de API-Sports (plan free)
REQUEST_LIMIT = 100
request_count = 0

# Rango de años a extraer (ajustable)
START_YEAR = 2022
END_YEAR = 2024

TEAMS = [
    # Grupo A
    "Mexico", "South Korea", "South Africa", "Czech Republic",
    # Grupo B
    "Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland",
    # Grupo C
    "Brazil", "Morocco", "Haiti", "Scotland",
    # Grupo D
    "USA", "Australia", "Paraguay", "Turkey",
    # Grupo E
    "Germany", "Ecuador", "Ivory Coast", "Curacao",
    # Grupo F
    "Netherlands", "Japan", "Sweden", "Tunisia",
    # Grupo G
    "Belgium", "Iran", "Egypt", "New Zealand",
    # Grupo H
    "Spain", "Uruguay", "Saudi Arabia", "Cape Verde",
    # Grupo I
    "France", "Senegal", "Iraq", "Norway",
    # Grupo J
    "Argentina", "Algeria", "Austria", "Jordan",
    # Grupo K
    "Portugal", "Colombia", "Uzbekistan", "Congo DR",
    # Grupo L
    "England", "Croatia", "Ghana", "Panama"
]

In [2]:
# ============================================================
# 1. GROUP GENERATOR
# ============================================================

def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

TEAM_GROUPS = list(chunk_list(TEAMS, 4))

client = APIFootballClient()
team_id_cache = {}

In [3]:
# ============================================================
# 2. CONTADOR DE REQUESTS
# ============================================================

def counted_request(func, *args, **kwargs):
    global request_count

    if request_count >= REQUEST_LIMIT:
        print(f"\n🛑 Límite diario alcanzado ({REQUEST_LIMIT} requests). Deteniendo extracción.\n")
        return None

    result = func(*args, **kwargs)
    request_count += 1

    print(f"   🔢 Request #{request_count}/{REQUEST_LIMIT}")

    return result

In [4]:
# ============================================================
# 3. DASHBOARD
# ============================================================

def print_dashboard(group_idx, total_groups, team_idx, group, team_name, year):
    print("\n--------------------------------------------------")
    print(f"📊 Progreso")
    print(f"   Grupo: {group_idx+1}/{total_groups}")
    print(f"   Equipo en grupo: {team_idx+1}/{len(group)} — {team_name}")
    print(f"   Año: {year}")
    print(f"   Requests usados: {request_count}/{REQUEST_LIMIT}")
    print(f"   Requests restantes: {max(0, REQUEST_LIMIT - request_count)}")
    print("--------------------------------------------------\n")

In [6]:

# ============================================================
# 4. FUNCIONES ROBUSTAS
# ============================================================

def get_team_id_cached(team_name):
    if team_name in team_id_cache:
        return team_id_cache[team_name]

    team_id = counted_request(client.get_national_team_id, team_name)

    if not team_id or team_id == 0:
        print(f"⚠️ ID inválido para {team_name}. Reintentando...")
        team_id = retry_team_id(team_name)

    team_id_cache[team_name] = team_id
    return team_id


def retry_team_id(team_name, retries=3):
    for i in range(retries):
        sleep_time = 2 ** i
        print(f"   ↳ Retry ID {i+1}/{retries} — {sleep_time}s...")
        time.sleep(sleep_time)

        team_id = counted_request(client.get_national_team_id, team_name)
        if team_id and team_id != 0:
            print(f"   ✔ ID recuperado: {team_id}")
            return team_id

    print(f"   ❌ No se pudo obtener ID válido para {team_name}")
    return None


def safe_get_fixtures(team_id, year, retries=3):
    for i in range(retries):
        data = counted_request(client.get, "fixtures", {"team": team_id, "season": year})

        if data and "response" in data and len(data["response"]) > 0:
            return data["response"]

        sleep_time = 2 ** i
        print(f"   ↳ Fixtures vacíos. Retry {i+1}/{retries} — {sleep_time}s...")
        time.sleep(sleep_time)

    print(f"   ❌ No se pudieron obtener fixtures para {team_id} en {year}")
    return []

In [7]:
# ============================================================
# 5. EXTRACCIÓN PRINCIPAL
# ============================================================

def extract_group(group_idx, group, total_groups):
    all_fixtures = []

    print(f"\n==============================")
    print(f"   Extrayendo grupo {group_idx+1}/{total_groups}: {group}")
    print(f"==============================\n")

    for team_idx, team in enumerate(group):
        if request_count >= REQUEST_LIMIT:
            break

        print(f"➡️ Equipo: {team}")

        team_id = get_team_id_cached(team)
        if not team_id:
            print(f"\n🛑 Stop general: no se pudo obtener ID para {team}.")
            print(f"   Última posición: grupo {group_idx+1}, equipo {team_idx+1} ({team}).")
            print("   Reanuda desde este punto cuando se refresque el límite diario.\n")
            # Devolvemos lo que llevamos del grupo para no perder nada
            return all_fixtures, True  # True = stop_global

        print(f"   ✔ ID encontrado: {team_id}")

        for year in range(START_YEAR, END_YEAR + 1):
            if request_count >= REQUEST_LIMIT:
                break

            print_dashboard(group_idx, total_groups, team_idx, group, team, year)
            print(f"   📅 Año {year}...")

            fixtures = safe_get_fixtures(team_id, year)
            all_fixtures.extend(fixtures)

            time.sleep(random.uniform(0.8, 1.5))

    return all_fixtures, False  # False = no stop_global

In [8]:
# ============================================================
# 6. GUARDADO
# ============================================================

def save_group_fixtures(group_index, fixtures):
    if not fixtures:
        print(f"\nℹ️ Grupo {group_index+1}: sin fixtures para guardar.\n")
        return

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{OUTPUT_DIR}/fixtures_group_{group_index+1}_{timestamp}.json"

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(fixtures, f, indent=2)

    print(f"\n💾 Guardado: {filename}\n")

In [ ]:
print("🚀 Iniciando extracción local...\n")
total_groups = len(TEAM_GROUPS)
global_stop = False
for idx, group in enumerate(TEAM_GROUPS):
    if request_count >= REQUEST_LIMIT or global_stop:
        break
    fixtures, stop_flag = extract_group(idx, group, total_groups)
    save_group_fixtures(idx, fixtures)
    if stop_flag:
        global_stop = True
        break
print("\n🎉 Extracción completada (o detenida por límite/ID).")
print(f"   Requests usados: {request_count}/{REQUEST_LIMIT}")
print("   Revisa los archivos en la carpeta raw_fixtures.\n")